In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pathlib

%matplotlib inline
%load_ext autoreload
%autoreload 2

style_file = pathlib.Path.cwd().parent / "plot_style.mplstyle"
plt.style.use(["default", style_file])

inputpath_data = pathlib.Path.cwd() / 'input_data/behavior' 
inputpath_data.mkdir(parents=True, exist_ok=True)
savepath_data = pathlib.Path.cwd() / 'output_data/behavior' 
savepath_data.mkdir(parents=True, exist_ok=True)
savepath_fig = pathlib.Path.cwd() / 'output_fig/behavior' 
savepath_fig.mkdir(parents=True, exist_ok=True)

cm_inch = 1/2.54

%config InlineBackend.figure_format='retina'

palette_genotypes = {'R39H12': '#000000', 'rdl_RNAi': '#828288', 'R39H12>rdl_RNAi': '#52c3c3', 'ort_RNAi': '#828288', 'R39H12>ort_RNAi': '#e8872e', 'rdl_RNAi_ort_RNAi': '#828288', 'R39H12>rdl_RNAi_ort_RNAi': '#662864'}
genotype_formatting = {'R39H12': 'R39H12', 'rdl_RNAi': 'Rdl RNAi', 'R39H12>rdl_RNAi': '#52c3c3', 'ort_RNAi': '#828288', 'R39H12>ort_RNAi': '#e8872e', 'rdl_RNAi_ort_RNAi': '#828288', 'R39H12>rdl_RNAi_ort_RNAi': '#662864'}

# Edge stimulus: 60 deg/s, screen 1 (b-f) = 0.9s - 2.4s, screen 2 (center) = 2.4s - 3.9s, screen 3 (f-b) = 3.9s - 5.4s
# Edge at 0 deg = 3.15 s. Central (binocular) visual field (10 deg left, 10 deg right): 2.983s - 3.317s 
# Chosen back-to-front time window: +- 90 deg to +- 10 deg: 1.65s - 2.983s
# Chosen front-to-back time window: +- 90 deg to +- 10 deg: 3.317s - 4.65s


In [ ]:
data1_edges = pd.read_csv(inputpath_data / 'behavior_preprocessed_open_loop_ON_OFF_edges.csv')
data2_barfixation = pd.read_csv(inputpath_data / 'behavior_preprocessed_closed_loop_barfixation.csv')

# Apply inclusion criteria to datasets

In [ ]:
def include_exclude(data, forward_min, min_trials, r_dev=None):
    """ Apply inclusion criteria to preprocessed dataset
    
        :data: pandas dataframe.
        :forward_min: the minimal average forward walking velocity (cm/s) for which a trial is included.
        :r_dev: the maximal average rotational velocity (degrees/s) over all included trials for which a fly is included (only used in fixation experiments).
        :min_trials: the minimal number of trials for which a fly is included.
        
        :returns: only the included trials and flies.
    """
    
    # Calculate mean forward speed per fly per trial
    d = data
    mean_forward = d.groupby(['flyname', 'trial']).forward.mean()
    d = d.merge(mean_forward, on=['flyname', 'trial'], suffixes=('', '_mean'))

    ###################### INCLUSION CRITERION (mean forward velocity) ######################
    fast_data = d.query("forward_mean >= @forward_min")
    
    curr_data = fast_data
    n_trials = curr_data.groupby(['flyname']).trial.unique().str.len()
    curr_data = curr_data.merge(n_trials, on='flyname', suffixes=('', 's_n'))
    s = curr_data
    
    mean_rotation = s.groupby(['flyname']).rotation.mean()
    s = s.merge(mean_rotation, on=['flyname'], suffixes=('', '_mean'))

    if r_dev != None:
        ######### INCLUSION CRITERION (average rotation deviation from 0 degree) ############
        straight_flies = s.query("rotation_mean <= (0 + @r_dev) and rotation_mean >= (0 - @r_dev)")
    else:
        straight_flies = s

    ##################### INCLUSION CRITERION (minimal number of trials) ####################
    included_data = straight_flies.query("trials_n >= @min_trials")

    data_included = included_data.drop(columns=['forward_mean', 'rotation_mean'])
    
    return data_included

In [ ]:
data1_edges_included = include_exclude(data = data1_edges, forward_min = 0.15, min_trials = 10)
data2_barfixation_included = include_exclude(data = data2_barfixation, forward_min = 0.30, min_trials = 50, r_dev = 10)

In [ ]:
def order_dataset(data, columns_to_keep, index_columns):
    """ Order dataset based on genotypes and remove unnecessary columns
    
    :data: dataset with all included flies and trials
    :columns_to_keep: list of all columns that should be kept
    :index_columns: list of all columns that should be used as index
    
    :return: ordered dataset
    """
    
    data = data.loc[:, columns_to_keep]
    genotype_list = {
        'R39H12': 1,
        'rdl_RNAi': 2,
        'R39H12>rdl_RNAi': 3,
        'ort_RNAi': 4,
        'R39H12>ort_RNAi': 5,
        'rdl_RNAi_ort_RNAi': 6,
        'R39H12>rdl_RNAi_ort_RNAi': 7,
    }
    
    data.loc[:,'ordered_index'] = data['genotype'].map(genotype_list)
    indices = ['ordered_index'] + index_columns
    data_ordered = data.set_index(indices).sort_values(by=indices) 
    
    return data_ordered

In [ ]:
data1_edges_ordered = order_dataset(data1_edges_included, columns_to_keep = ['genotype', 'flyname', 'brighterdarker', 
                                        'edge_rotation_in_unitcircle_conventions', 'trial', 
                                        'time', 'forward', 'rotation', 'x', 'y'], index_columns = ['genotype', 'flyname', 
                                        'brighterdarker', 'edge_rotation_in_unitcircle_conventions', 
                                                                              'trial', 'time'])

In [ ]:
data2_barfixation_ordered = order_dataset(data2_barfixation_included, columns_to_keep = ['genotype', 'flyname', 'trial', 
                                        'time', 'forward', 'rotation', 'bar_angle', 'trials_n'], index_columns = ['genotype', 'flyname', 
                                                                                         'trial', 'time'])

# Rotated edges open-loop experiment, with T4/T5 R39H12-Gal4, UAS-ort_RNAi, UAS-rdl_RNAi and both

## Normalise rotation per trial: set mean rotation of every trial (left and right motion) to 0

In [ ]:
# Calculate mean rotation per fly per trial
temp = data1_edges_ordered.reset_index()
rotation_mean = temp.groupby(['flyname', 'trial']).rotation.mean()
temp = temp.merge(rotation_mean, on=['flyname', 'trial'], suffixes=('', '_mean'))

# Recalculate rotation based on trial mean. 
data1_edges_rotation_normalized = temp
data1_edges_rotation_normalized.at[:,'rotation_normalized'] = data1_edges_rotation_normalized.rotation - data1_edges_rotation_normalized.rotation_mean
data1_edges_rotation_normalized['rotation_in_stimdir'] = data1_edges_rotation_normalized.apply(lambda row: row.rotation_normalized * -1 if row.edge_rotation_in_unitcircle_conventions == 180 else row.rotation_normalized, axis=1)
# Average the fly's rotation over trials
data1_edges_rotation_trial_averaged = pd.DataFrame(data1_edges_rotation_normalized.groupby(['ordered_index', 'genotype', 'flyname', 'brighterdarker','time']).rotation_in_stimdir.mean())
data1_edges_rotation_trial_averaged.reset_index().groupby(["ordered_index", "genotype", "brighterdarker"]).apply(lambda df: len(df.flyname.unique()))

In [ ]:
quantify = data1_edges_rotation_trial_averaged.reset_index()

bf_peak = np.array(quantify.query("time > 1.65 and time < 2.983").groupby(['ordered_index', 'genotype', 'flyname', 'brighterdarker']).rotation_in_stimdir.idxmax())

bf_before_peak = bf_peak - 1
bf_after_peak = bf_peak + 1

bf_around_peak = pd.concat([quantify.loc[bf_peak], quantify.loc[bf_before_peak], quantify.loc[bf_after_peak]]).sort_index()
bf_peak_avg = pd.DataFrame(bf_around_peak.groupby(['ordered_index', 'genotype', 'flyname', 'brighterdarker']).rotation_in_stimdir.mean()).rename(columns={'rotation_in_stimdir':'peak_rotation'})
bf_peak_avg['edge_location'] = 'bf'

fb_peak = np.array(quantify.query("time > 3.317 and time < 4.65").groupby(['ordered_index', 'genotype', 'flyname', 'brighterdarker']).rotation_in_stimdir.idxmax())
fb_before_peak = fb_peak - 1
fb_after_peak = fb_peak + 1

fb_around_peak = pd.concat([quantify.loc[fb_peak], quantify.loc[fb_before_peak], quantify.loc[fb_after_peak]]).sort_index()
fb_peak_avg = pd.DataFrame(fb_around_peak.groupby(['ordered_index', 'genotype', 'flyname', 'brighterdarker']).rotation_in_stimdir.mean()).rename(columns={'rotation_in_stimdir':'peak_rotation'})
fb_peak_avg['edge_location'] = 'fb'

peak_rotations = pd.concat([bf_peak_avg, fb_peak_avg], axis=0)
peak_averages_genotypes = pd.DataFrame(peak_rotations.groupby(['ordered_index', 'genotype', 'brighterdarker', 'edge_location']).peak_rotation.mean()).reset_index()

In [ ]:
genotypes_n_edges = pd.DataFrame(data1_edges_rotation_trial_averaged.reset_index().groupby(['ordered_index', 'genotype']).flyname.unique().apply(lambda row: len(row))).rename(columns={'flyname':'n_flies'}).reset_index()

In [ ]:
# Stats in Graphpad Prism
behavior_peak_rotations = peak_rotations.reset_index().copy()
behavior_peak_rotations['order_genotype'] = behavior_peak_rotations.apply(lambda row: str(row.ordered_index) + '_' + row.genotype + '_' + row.edge_location, axis=1)
behavior_peak_rotations_wide = behavior_peak_rotations.pivot(index='flyname', columns='order_genotype', values='peak_rotation').reset_index().drop(columns='flyname')
behavior_peak_rotations_wide = behavior_peak_rotations_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
behavior_peak_rotations_wide_bf = behavior_peak_rotations_wide.iloc[:,np.arange(0,14,2)]
behavior_peak_rotations_wide_fb = behavior_peak_rotations_wide.iloc[:,np.arange(1,15,2)]
behavior_peak_rotations_wide_fb_minus_bf = pd.DataFrame(behavior_peak_rotations_wide_fb.values - behavior_peak_rotations_wide_bf.values, columns=behavior_peak_rotations_wide_bf.columns).rename(columns = lambda x : str(x)[:-3])
behavior_peak_rotations_wide_fb_minus_bf.to_csv(savepath_data / 'behavior_edges_peak_rotation_fb_minus_bf.csv', sep=',', float_format='%.5f')

## Bar fixation

In [ ]:
data2_barfixation_ordered

In [ ]:
print("Bar fixation R39H12")
for gt in data2_barfixation_ordered.reset_index().genotype.unique():
    print(gt, "n =", len(data2_barfixation_ordered.reset_index().query("genotype == @gt").flyname.unique()))

In [ ]:
data2_barfixation_densityplots = data2_barfixation_ordered.groupby(['ordered_index','genotype','flyname', 'trial', 'trials_n']).agg({'bar_angle':list})
data2_barfixation_densityplots = data2_barfixation_densityplots.reset_index()

In [ ]:
data2_barfixation_densityplots

Divide the bar_angle range [-180 - 180 degrees] into 72 histogram bins spanning 5 degrees each, and count for every trial how many measurements fall into each bin. The total hist_count for each trial will sum to the 400 measurements of which every trial in our experiment consists of.

In [ ]:
def bar_angle_histogram_counts(data, measurement_range_min, measurement_range_max, bins):  
    """
    Count the number of values per value that fall in each defined histogram bin.
    data: dataset
    measurement_range_min: minimal value of the measurement range (will be lower bound of histogram).
    measurement_range_max: maximal value of the measurement range (will be upper bound of histogram).
    bins: number of bins to divide the measurement_range in. 
    """
    hist_counts = pd.DataFrame(columns=['ordered_index', 'genotype', 'flyname', 'trial', 'hist_counts'])

    for i, fly in enumerate(data.flyname):
        hist_count = np.histogram(data.loc[i, 'bar_angle'][:], range = (measurement_range_min, measurement_range_max), bins = bins)[0]
        trial = data.loc[i, 'trial']
        genotype = data.loc[i, 'genotype']
        ordered_index = data.loc[i, 'ordered_index']
        temp = pd.DataFrame([[ordered_index, genotype, fly, trial, hist_count]], columns=(['ordered_index', 'genotype', 'flyname', 'trial', 'hist_counts']))
        hist_counts = pd.concat([hist_counts, temp])
    return hist_counts.reset_index(drop=True)

In [ ]:
data2_barfixation_hist_counts = bar_angle_histogram_counts(data2_barfixation_densityplots, -180, 180, 72)

In [ ]:
data2_barfixation_hist_counts

In [ ]:
def expand_hist_counts(data, measurement_range_min, measurement_range_max, bins):
    """
    data: dataset
    measurement_range_min: minimal value of the measurement range (will be lower bound of histogram).
    measurement_range_max: maximal value of the measurement range (will be upper bound of histogram).
    bins: number of bins to divide the measurement_range in. 
    
    Give the same values as used in the bar_angle_histogram_counts function.
    """
    
    hist_counts_flyagg = data.set_index(['ordered_index', 'genotype', 'flyname', 'trial'])
    hist_counts_flyagg = hist_counts_flyagg.groupby(['ordered_index', 'genotype', 'flyname']).agg({'hist_counts':list})
    hist_counts_flyagg = hist_counts_flyagg.reset_index()

    mean_bin_counts = pd.DataFrame(columns = (['ordered_index', 'genotype', 'flyname', 'mean_bin_count']))

    for i, fly in enumerate(hist_counts_flyagg.flyname):
        flydata = hist_counts_flyagg.loc[i, 'hist_counts']
        mean_bin_count = [float(sum(col))/len(col) for col in zip(*flydata)] # average over trials (rows) for all bins (columns).
        ordered_index = hist_counts_flyagg.loc[i, 'ordered_index']
        genotype = hist_counts_flyagg.loc[i, 'genotype']
        temp = pd.DataFrame([[ordered_index, genotype, fly, mean_bin_count]], columns=(['ordered_index', 'genotype', 'flyname', 'mean_bin_count']))
        mean_bin_counts = pd.concat([mean_bin_counts, temp])
  
    bin_size = (measurement_range_max - measurement_range_min) / bins
    
    mean_bin_counts_long = mean_bin_counts.explode('mean_bin_count').reset_index(drop=True)
    bin_midpoint_ = [*np.arange((measurement_range_min + 0.5 * bin_size), (measurement_range_max + 0.5 * bin_size), bin_size)] # midpoint values of all bins
    mean_bin_counts_long['bin_midpoint'] = np.resize(bin_midpoint_,len(mean_bin_counts_long))
    mean_bin_counts_long.mean_bin_count = mean_bin_counts_long.mean_bin_count.astype(float)
    mean_bin_counts_long.bin_midpoint = mean_bin_counts_long.bin_midpoint.astype(float)
    
    return mean_bin_counts_long

In [ ]:
data2_barfixation_hist_counts_expanded = expand_hist_counts(data2_barfixation_hist_counts, -180, 180, 72)

# To go from counts per 5°-bin to probability densities in % per °, we divide the values by 20. Since 400 values per trial
# divided by 4 gives percentages for each 5°-bin, and dividing by 5 gives the values per °. 
data2_barfixation_hist_densities = data2_barfixation_hist_counts_expanded
data2_barfixation_hist_densities.mean_bin_count = data2_barfixation_hist_densities.mean_bin_count / 20
data2_barfixation_hist_densities = data2_barfixation_hist_densities.rename(columns={'mean_bin_count': 'probability_density'})

## Probability quantification

In [ ]:
def fixation_in_front(data, measurement_range_min, measurement_range_max, bins):
    """
    data: dataset
    measurement_range_min: minimal value of the measurement range (will be lower bound of histogram).
    measurement_range_max: maximal value of the measurement range (will be upper bound of histogram).
    bins: number of bins to divide the measurement_range in. 
    
    Give the same values as used in the bar_angle_histogram_counts function.
    """
    
    bin_size = (measurement_range_max - measurement_range_min) / bins
    bin_midpoint_ = [*np.arange((measurement_range_min + 0.5 * bin_size), (measurement_range_max + 0.5 * bin_size), bin_size)] # midpoint values of all bins
    
    hist_counts_per_trial_long = data.explode('hist_counts')
    hist_counts_per_trial_long['bin_midpoint'] = np.resize(bin_midpoint_,len(hist_counts_per_trial_long))
    hist_counts_per_trial_long.hist_counts = hist_counts_per_trial_long.hist_counts.astype(float)
    hist_counts_per_trial_long.bin_midpoint = hist_counts_per_trial_long.bin_midpoint.astype(float)
    
    prob_density = hist_counts_per_trial_long.query("bin_midpoint in [-27.5, -22.5, -17.5, -12.5, -7.5, -2.5, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5]")

    # divide hist_counts by 400 to make all values of one trial add up to 1 instead of 400.
    prob_density['prob_per_bin'] = prob_density.hist_counts / 400 

    fixation_in_front = pd.DataFrame(prob_density.groupby(['ordered_index','genotype','flyname', 'trial']).prob_per_bin.sum())
    fixation_in_front = fixation_in_front.rename({"prob_per_bin":"fixation_in_front"}, axis = 1)
    fixation_in_front_flymeans = pd.DataFrame(fixation_in_front.reset_index().groupby(['ordered_index','genotype','flyname']).fixation_in_front.mean())
    fixation_in_front_plotting = fixation_in_front_flymeans.reset_index()
    
    fixation_in_front_plotting["percent"] = fixation_in_front_plotting.fixation_in_front * 100
    
    return fixation_in_front_plotting

In [ ]:
data2_barfixation_fixation_in_front = fixation_in_front(data2_barfixation_hist_counts, -180, 180, 72)

behavior_fixation_in_front = data2_barfixation_fixation_in_front.reset_index().copy()
behavior_fixation_in_front['order_genotype'] = behavior_fixation_in_front.apply(lambda row: str(row.ordered_index) + '_' + row.genotype, axis=1)
behavior_fixation_in_front_wide = behavior_fixation_in_front.pivot(index='flyname', columns='order_genotype', values='percent').reset_index().drop(columns='flyname')
behavior_fixation_in_front_wide = behavior_fixation_in_front_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
behavior_fixation_in_front_wide.to_csv(savepath_data / 'behavior_bars_fixation_in_front.csv', sep=',', float_format='%.5f')

In [ ]:
genotypes_n_fixation = pd.DataFrame(data2_barfixation_fixation_in_front.groupby(['ordered_index', 'genotype']).flyname.unique().apply(lambda row: len(row))).rename(columns={'flyname':'n_flies'}).reset_index()
genotypes_n_fixation

In [ ]:
fixation_avg = pd.DataFrame(data2_barfixation_fixation_in_front.groupby(['ordered_index', 'genotype']).percent.mean()).reset_index()

In [ ]:
plot_data = data1_edges_rotation_trial_averaged.reset_index()

plotting_parameters = [
    
    {'genotype': 'rdl_RNAi', 'axis': 'AD'},
    {'genotype': 'R39H12', 'axis': 'AD'},
    {'genotype': 'R39H12>rdl_RNAi', 'axis': 'AD'},
    
    {'genotype': 'ort_RNAi', 'axis': 'BE'},
    {'genotype': 'R39H12', 'axis': 'BE'},
    {'genotype': 'R39H12>ort_RNAi', 'axis': 'BE'},
    
    {'genotype': 'rdl_RNAi_ort_RNAi', 'axis': 'CF'},
    {'genotype': 'R39H12', 'axis': 'CF'},
    {'genotype': 'R39H12>rdl_RNAi_ort_RNAi', 'axis': 'CF'},
]

plotting_parameters_fixation = [
    {'genotype': 'rdl_RNAi', 'axis': 'G'},
    {'genotype': 'R39H12', 'axis': 'G'},
    {'genotype': 'R39H12>rdl_RNAi', 'axis': 'G'},
    
    {'genotype': 'ort_RNAi', 'axis': 'H'},
    {'genotype': 'R39H12', 'axis': 'H'},
    {'genotype': 'R39H12>ort_RNAi', 'axis': 'H'},
    
    {'genotype': 'rdl_RNAi_ort_RNAi', 'axis': 'I'},
    {'genotype': 'R39H12', 'axis': 'I'},
    {'genotype': 'R39H12>rdl_RNAi_ort_RNAi', 'axis': 'I'},
]

plotting_parameters_fixation_q = [
    {'genotype': 'R39H12', 'axis': 'J'},
    {'genotype': 'rdl_RNAi', 'axis': 'J'},
    {'genotype': 'R39H12>rdl_RNAi', 'axis': 'J'},
    
    {'genotype': 'R39H12', 'axis': 'K'},
    {'genotype': 'ort_RNAi', 'axis': 'K'},
    {'genotype': 'R39H12>ort_RNAi', 'axis': 'K'},
    
    {'genotype': 'R39H12', 'axis': 'L'},
    {'genotype': 'rdl_RNAi_ort_RNAi', 'axis': 'L'},
    {'genotype': 'R39H12>rdl_RNAi_ort_RNAi', 'axis': 'L'},
]

fig, axd = plt.subplot_mosaic(
    'ADGJ;BEHK;CFIL',
    sharex=False,
    sharey=False,
    figsize=(14 * cm_inch, 10 * cm_inch),
    dpi=300,
    gridspec_kw={
        "wspace": 0.4,
        "hspace": 0.3,
        "width_ratios": [1,0.5,1,0.5]
    },
)

for subgroup in plotting_parameters:
    genotype = subgroup['genotype']
    axis = subgroup['axis'][0]
    n_flies = len(plot_data.query("genotype in @genotype").flyname.unique())

    subgroup_data = plot_data.query("genotype in @genotype")
    sns.lineplot(data=subgroup_data, x='time', y='rotation_in_stimdir', errorbar='se', hue='genotype', palette=palette_genotypes, ax=axd[axis], legend=None, lw=1, err_kws={'alpha':0.3, 'lw':0});
    
    axis = subgroup['axis'][1]
    sns.lineplot(data=peak_rotations.query("genotype in @genotype"), x='edge_location', y='peak_rotation', markers=True, marker='o', markeredgewidth=0, markersize=2.3, hue='genotype', palette=palette_genotypes, legend=False, errorbar='se', err_kws={'elinewidth':0.75, 'capsize':1, 'capthick':0.75}, err_style='bars', estimator='mean', linewidth=1, ax=axd[axis])#, style='brighterdarker');
    
for subgroup in plotting_parameters_fixation:
    genotype = subgroup['genotype']
    axis = subgroup['axis']
    subgroup_data = data2_barfixation_hist_densities.query("genotype in @genotype")
    sns.lineplot(data=subgroup_data, x='bin_midpoint', y='probability_density', errorbar='se', hue='genotype', palette=palette_genotypes, ax=axd[axis], legend=None, lw=1, err_kws={'alpha':0.3, 'lw':0});
    
for subgroup in plotting_parameters_fixation_q:
    np.random.seed(123)
    genotype = subgroup['genotype']
    axis = subgroup['axis']
    subgroup_data = data2_barfixation_fixation_in_front.query("genotype in @genotype")
    averages_data = fixation_avg.query("genotype in @genotype")
    sns.stripplot(data=subgroup_data, x='genotype', y='percent', palette=palette_genotypes, hue='genotype', alpha=0.3, s=3, jitter=0.1, dodge=False, edgecolor='white', linewidth=0.2, ax=axd[axis], legend=None);
    sns.scatterplot(data=averages_data, x='genotype', y='percent', palette=palette_genotypes, hue='genotype', alpha=1, ax=axd[axis], legend=None, zorder=3, marker='_', s=120, lw=1.5);
    
for axis in 'ABCDEFGHIJKL':
    axd[axis].spines[['top', 'right']].set_visible(False)
    axd[axis].set_ylabel('')
    
for axis in 'ABCDEF':
    axd[axis].set_ylim([-10,100])
    axd[axis].set_yticks([0, 50, 100])
    
for axis in 'ABC':
    axd[axis].fill_between([1.65, 2.983], [-10, -10], [100, 100], color='k', alpha=0.2, linewidth=0, zorder=-10)
    axd[axis].fill_between([3.317, 4.65], [-10, -10], [100, 100], color='k', alpha=0.2, linewidth=0, zorder=-10)
    axd[axis].set_xticks([0, 2, 4, 6])
    axd[axis].set_xlabel('Time (s)')
    axd[axis].set_ylabel('Angular velocity (° s-1)')
    
for row in genotypes_n_edges.itertuples():
    axd['A'].text(0, 200-15*row.Index, row.genotype + ', n = ' + str(row.n_flies), color=palette_genotypes[row.genotype], fontsize=6, zorder=100)

for row in genotypes_n_fixation.itertuples():
    axd['D'].text(0, 200-15*row.Index, row.genotype + ', n = ' + str(row.n_flies), color=palette_genotypes[row.genotype], fontsize=6, zorder=100)
    
for axis in 'ABDEGH':
    axd[axis].spines[['bottom']].set_visible(False)
    axd[axis].xaxis.set_visible(False)
    
for axis in 'DEF':
    axd[axis].margins(x=0.4)
    axd[axis].set_xlabel('')
    axd[axis].set_ylabel('Peak velocity (° s-1)')
    
for axis in 'GHI':
    axd[axis].set_xlabel('Bar position (°)')
    axd[axis].fill_between([-30, 30], [0, 0], [90, 90], color='k', alpha=0.2, linewidth=0, zorder=-10)
    axd[axis].set_xticks([-180, -90, 0, 90, 180])
    axd[axis].set_yticks([0, 0.2, 0.4, 0.6, 0.8])
    axd[axis].set_ylim([0, 0.9])
    axd[axis].set_ylabel('Probability density (% per °)')
    
for axis in 'JKL':
    axd[axis].set_xlabel('')
    axd[axis].set_yticks([0, 20, 40, 60])
    axd[axis].set_ylim([0, 70])
    axd[axis].margins(x=0.2)
    axd[axis].axhline(16.6667, color = 'k', linestyle=':')
    axd[axis].set_ylabel('Fixation in front (%)')

figtitle = "behavior_edges_mean_sem_fixation_mean_sem.pdf"
plt.savefig(savepath_fig / figtitle)